# LoRA vs Full Fine-Tuning on RoBERTa-base

This notebook clones the project repo, installs dependencies, and runs all four
experiments:

| Mode | Dataset |
|---|---|
| baseline | SST-2 |
| lora | SST-2 |
| baseline | MNLI |
| lora | MNLI |

Results are collected and printed as a comparison table at the end.

## 1. Setup

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
REPO_URL      = "https://github.com/da-luce/cs5782_final_project.git"
BRANCH        = "jason_LoRA_impl"   # change to "main" for the final submission
TRAIN_SAMPLES = 5000                # training samples per experiment
VAL_SAMPLES   = 500                 # validation samples per experiment
EPOCHS        = 5                   # training epochs per experiment
# ────────────────────────────────────────────────────────────────────────────

In [ ]:
REPO_DIR = REPO_URL.split("/")[-1].replace(".git", "")

!git clone -b {BRANCH} {REPO_URL}
%cd {REPO_DIR}

In [ ]:
!pip install -r requirements.txt -q

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 2. SST-2 Experiments

In [ ]:
!python code/train.py \
    --mode baseline \
    --dataset sst2 \
    --train_samples {TRAIN_SAMPLES} \
    --val_samples {VAL_SAMPLES} \
    --epochs {EPOCHS}

In [ ]:
!python code/train.py \
    --mode lora \
    --dataset sst2 \
    --train_samples {TRAIN_SAMPLES} \
    --val_samples {VAL_SAMPLES} \
    --epochs {EPOCHS}

## 3. MNLI Experiments

In [ ]:
!python code/train.py \
    --mode baseline \
    --dataset mnli \
    --train_samples {TRAIN_SAMPLES} \
    --val_samples {VAL_SAMPLES} \
    --epochs {EPOCHS}

In [ ]:
!python code/train.py \
    --mode lora \
    --dataset mnli \
    --train_samples {TRAIN_SAMPLES} \
    --val_samples {VAL_SAMPLES} \
    --epochs {EPOCHS}

## 4. Results Comparison

In [ ]:
import json
import os

experiments = [
    ("baseline", "sst2"),
    ("lora",     "sst2"),
    ("baseline", "mnli"),
    ("lora",     "mnli"),
]

rows = []
for mode, dataset in experiments:
    path = f"results/{mode}_{dataset}.json"
    if not os.path.exists(path):
        print(f"Missing results for {mode}/{dataset} — skipping")
        continue
    with open(path) as f:
        info = json.load(f)
    accuracy = info["eval_results"].get("eval_accuracy", float("nan"))
    trainable = info["trainable_params"]["trainable"]
    trainable_pct = info["trainable_params"]["trainable_pct"]
    total = info["trainable_params"]["total"]
    time_min = info["training_time_sec"] / 60
    rows.append((mode, dataset, accuracy, trainable, trainable_pct, total, time_min))

header = f"{'Mode':<12} {'Dataset':<8} {'Accuracy':>10} {'Trainable params':>18} {'Trainable %':>12} {'Total params':>14} {'Train time (min)':>18}"
sep = "-" * len(header)
print(header)
print(sep)
for mode, dataset, acc, tr, tr_pct, tot, t in rows:
    print(f"{mode:<12} {dataset:<8} {acc:>10.4f} {tr:>18,} {tr_pct:>11.4f}% {tot:>14,} {t:>17.1f}m")

In [ ]:
# Print raw JSON for each completed experiment so results are preserved in notebook output
import glob
for path in sorted(glob.glob("results/*.json")):
    print(f"\n{'='*60}\n{path}\n{'='*60}")
    with open(path) as f:
        print(f.read())

In [ ]:
# Download result JSONs to your local machine
from google.colab import files
import glob

for path in sorted(glob.glob("results/*.json")):
    files.download(path)